In [0]:
# BusLight - verificações finais
# 1) rodar o MERGE da Prata uma segunda vez e mostrar que nada novo entra
# 2) conferir a integridade da Ouro (chaves únicas e soma do fato igual à da Prata)

SCHEMA = "workspace.buslight"

## 1. Segunda execução do MERGE da Prata
Contagem antes, MERGE com o mesmo lote e contagem depois. Na segunda vez `num_inserted_rows` deve ser 0.

In [0]:
# clima por hora (chave: data_hora)
antes = spark.table(f"{SCHEMA}.prata_clima_hora").count()

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW lote_clima AS
SELECT to_timestamp(h.time) AS data_hora,
       CAST(h.temperature_2m AS DOUBLE) AS temperatura_c,
       COALESCE(CAST(h.precipitation AS DOUBLE), 0.0) AS precipitacao_mm,
       COALESCE(CAST(h.rain AS DOUBLE), 0.0) AS chuva_mm
FROM (SELECT explode(arrays_zip(dados.hourly.time, dados.hourly.temperature_2m,
                                dados.hourly.precipitation, dados.hourly.rain)) AS h
      FROM {SCHEMA}.bronze_clima)
WHERE h.time IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY h.time ORDER BY h.time) = 1
""")

resultado = spark.sql(f"""
MERGE INTO {SCHEMA}.prata_clima_hora AS alvo
USING lote_clima AS novo
ON alvo.data_hora = novo.data_hora
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
display(resultado)
depois = spark.table(f"{SCHEMA}.prata_clima_hora").count()
print(f"prata_clima_hora: {antes} linhas antes, {depois} depois")

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
48,45,0,3


prata_clima_hora: 48 linhas antes, 51 depois


In [0]:
# usuários (chave: id_usuario) - o mesmo teste com a tabela inteira como lote
antes = spark.table(f"{SCHEMA}.prata_usuario").count()

resultado = spark.sql(f"""
MERGE INTO {SCHEMA}.prata_usuario AS alvo
USING (SELECT * FROM {SCHEMA}.prata_usuario) AS novo
ON alvo.id_usuario = novo.id_usuario
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
display(resultado)
depois = spark.table(f"{SCHEMA}.prata_usuario").count()
print(f"prata_usuario: {antes} linhas antes, {depois} depois")

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
219,219,0,0


prata_usuario: 219 linhas antes, 219 depois


## 2. Integridade da Ouro

In [0]:
%sql
-- chave de cada dimensão é única (linhas = chaves distintas)
SELECT 'dim_ponto' AS dimensao, COUNT(*) AS linhas, COUNT(DISTINCT sk_ponto) AS chaves_distintas FROM workspace.buslight.dim_ponto
UNION ALL SELECT 'dim_linha', COUNT(*), COUNT(DISTINCT sk_linha) FROM workspace.buslight.dim_linha
UNION ALL SELECT 'dim_tempo', COUNT(*), COUNT(DISTINCT sk_tempo) FROM workspace.buslight.dim_tempo
UNION ALL SELECT 'dim_clima', COUNT(*), COUNT(DISTINCT sk_clima) FROM workspace.buslight.dim_clima
UNION ALL SELECT 'dim_usuario', COUNT(*), COUNT(DISTINCT sk_usuario) FROM workspace.buslight.dim_usuario

dimensao,linhas,chaves_distintas
dim_ponto,144,144
dim_linha,197,197
dim_tempo,8,8
dim_clima,48,48
dim_usuario,219,219


In [0]:
%sql
-- linhas do fato sem dimensão correspondente (deve ser 0 em todas)
SELECT
  SUM(CASE WHEN p.sk_ponto IS NULL THEN 1 ELSE 0 END) AS sem_ponto,
  SUM(CASE WHEN l.sk_linha IS NULL THEN 1 ELSE 0 END) AS sem_linha,
  SUM(CASE WHEN t.sk_tempo IS NULL THEN 1 ELSE 0 END) AS sem_tempo
FROM workspace.buslight.fato_espera_ponto f
LEFT JOIN workspace.buslight.dim_ponto p ON f.sk_ponto = p.sk_ponto
LEFT JOIN workspace.buslight.dim_linha l ON f.sk_linha = l.sk_linha
LEFT JOIN workspace.buslight.dim_tempo t ON f.sk_tempo = t.sk_tempo

sem_ponto,sem_linha,sem_tempo
0,0,0


In [0]:
%sql
-- a soma das métricas do fato é igual à da Prata
SELECT
  (SELECT COUNT(*) FROM workspace.buslight.prata_previsao_parada) AS previsoes_na_prata,
  (SELECT SUM(qtd_veiculos_previstos) FROM workspace.buslight.fato_espera_ponto) AS veiculos_no_fato,
  (SELECT COUNT(*) FROM workspace.buslight.fato_espera_ponto) AS linhas_do_fato

previsoes_na_prata,veiculos_no_fato,linhas_do_fato
25802,25802,9825
